# Algorithms

### Imports

In [26]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal

np.random.seed(12345)

### Data loading

In [27]:
DistanceMetric = Literal['euclidean', 'manhattan']
def get_dist_function(metric: DistanceMetric):
    if metric == 'euclidean':
        return lambda diff: np.sqrt((diff ** 2).sum(axis=-1))
    elif metric == 'manhattan':
        return lambda diff: np.abs(diff).sum(axis=-1)
    else:
        raise ValueError(f"Invalid metric: {metric}")

#### Capacitated Vehicle Routing Problem (CVRP)

In [28]:
class CvrpInstance:
    """
    Loads a CVRP instance from file.
    Format:
        line 1: vehicle capacity
        line 2 (depot): x, y
        remaining (customers): x, y, demand
    """
    def __init__(self, filepath, metric='euclidean'):
        with open(filepath, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]

        self.capacity = int(lines[0])
        self.depot = np.array(lines[1].split(), dtype=float)

        # shape (n, 3): columns are x, y, demand
        self.customers = np.loadtxt(lines[2:])

        # coords[0] = depot, coords[1..n] = customers
        self.coords = np.vstack([self.depot, self.customers[:, :2]])
        # demands[0] = 0 (depot), demands[1..n] = customer demands
        self.demands = np.concatenate([[0.0], self.customers[:, 2]])

        self.get_distance = get_dist_function(metric)
        self.dist_matrix = self._compute_distance_matrix()

    def _compute_distance_matrix(self):
        diff = self.coords[:, np.newaxis, :] - self.coords[np.newaxis, :, :]
        return self.get_distance(diff)
    
    def __repr__(self):
        return f"CvrpInstance(capacity={self.capacity}, depot={self.depot}, n={self.customers}, customers={self.customers})"

cvrp_data = CvrpInstance("instances/cvrp.txt")
print(cvrp_data)

CvrpInstance(capacity=160, depot=[30. 40.], n=[[37. 52.  7.]
 [49. 49. 30.]
 [52. 64. 16.]
 [20. 26.  9.]
 [40. 30. 21.]
 [21. 47. 15.]
 [17. 63. 19.]
 [31. 62. 23.]
 [52. 33. 11.]
 [51. 21.  5.]
 [42. 41. 19.]
 [31. 32. 29.]
 [ 5. 25. 23.]
 [12. 42. 21.]
 [36. 16. 10.]
 [52. 41. 15.]
 [27. 23.  3.]
 [17. 33. 41.]
 [13. 13.  9.]
 [57. 58. 28.]
 [62. 42.  8.]
 [42. 57.  8.]
 [16. 57. 16.]
 [ 8. 52. 10.]
 [ 7. 38. 28.]
 [27. 68.  7.]
 [30. 48. 15.]
 [43. 67. 14.]
 [58. 48.  6.]
 [58. 27. 19.]
 [37. 69. 11.]
 [38. 46. 12.]
 [46. 10. 23.]
 [61. 33. 26.]
 [62. 63. 17.]
 [63. 69.  6.]
 [32. 22.  9.]
 [45. 35. 15.]
 [59. 15. 14.]
 [ 5.  6.  7.]
 [10. 17. 27.]
 [21. 10. 13.]
 [ 5. 64. 11.]
 [30. 15. 16.]
 [39. 10. 10.]
 [32. 39.  5.]
 [25. 32. 25.]
 [25. 55. 17.]
 [48. 28. 18.]
 [56. 37. 10.]], customers=[[37. 52.  7.]
 [49. 49. 30.]
 [52. 64. 16.]
 [20. 26.  9.]
 [40. 30. 21.]
 [21. 47. 15.]
 [17. 63. 19.]
 [31. 62. 23.]
 [52. 33. 11.]
 [51. 21.  5.]
 [42. 41. 19.]
 [31. 32. 29.]
 [ 5. 25. 23

#### Vehicle Routing Problem with Time Windows (VRPTW):

In [29]:
class VrptwInstance(CvrpInstance):
    """
    Loads a VRPTW instance from file. Extends CvrpInstance.
    Format:
        line 1: num_vehicles, vehicle capacity
        line 2 (depot): x, y
        remaining (customers): x, y, demand, tw_open, tw_close, service_time
    """
    def __init__(self, filepath, metric='euclidean'):
        with open(filepath, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]

        header = lines[0].split()
        self.num_vehicles = int(header[0])
        self.capacity = int(header[1])
        self.depot = np.array(lines[1].split(), dtype=float)

        # shape (n, 6): columns are x, y, demand, tw_open, tw_close, service_time
        self.customers = np.loadtxt(lines[2:])

        self.coords = np.vstack([self.depot, self.customers[:, :2]])
        self.demands = np.concatenate([[0.0], self.customers[:, 2]])

        # index 0 = depot (no constraint), 1..n = customers
        self.tw_open = np.concatenate([[0.0], self.customers[:, 3]])
        self.tw_close = np.concatenate([[np.inf], self.customers[:, 4]])
        self.service_time = np.concatenate([[0.0], self.customers[:, 5]])

        self.get_distance = get_dist_function(metric)
        self.dist_matrix = self._compute_distance_matrix()
    
    def __repr__(self):
        return f"VrptwInstance(num_vehicles={self.num_vehicles}, capacity={self.capacity}, depot={self.depot}, n={self.customers.shape[0]}, customers={self.customers})"

vrptw_data = VrptwInstance("instances/vrptw.txt")
print(vrptw_data)

VrptwInstance(num_vehicles=25, capacity=200, depot=[35. 35.], n=100, customers=[[ 41.  49.  10. 161. 171.  10.]
 [ 35.  17.   7.  50.  60.  10.]
 [ 55.  45.  13. 116. 126.  10.]
 [ 55.  20.  19. 149. 159.  10.]
 [ 15.  30.  26.  34.  44.  10.]
 [ 25.  30.   3.  99. 109.  10.]
 [ 20.  50.   5.  81.  91.  10.]
 [ 10.  43.   9.  95. 105.  10.]
 [ 55.  60.  16.  97. 107.  10.]
 [ 30.  60.  16. 124. 134.  10.]
 [ 20.  65.  12.  67.  77.  10.]
 [ 50.  35.  19.  63.  73.  10.]
 [ 30.  25.  23. 159. 169.  10.]
 [ 15.  10.  20.  32.  42.  10.]
 [ 30.   5.   8.  61.  71.  10.]
 [ 10.  20.  19.  75.  85.  10.]
 [  5.  30.   2. 157. 167.  10.]
 [ 20.  40.  12.  87.  97.  10.]
 [ 15.  60.  17.  76.  86.  10.]
 [ 45.  65.   9. 126. 136.  10.]
 [ 45.  20.  11.  62.  72.  10.]
 [ 45.  10.  18.  97. 107.  10.]
 [ 55.   5.  29.  68.  78.  10.]
 [ 65.  35.   3. 153. 163.  10.]
 [ 65.  20.   6. 172. 182.  10.]
 [ 45.  30.  17. 132. 142.  10.]
 [ 35.  40.  16.  37.  47.  10.]
 [ 41.  37.  16.  39.  49.  10

### Fitness / cost evaluation

In [30]:
Route = np.ndarray
DistanceMatrix = np.ndarray
Solution = list[Route]

def route_cost(route: Route, dist_matrix: DistanceMatrix) -> float:
    """Total distance of a single route (depot -> customers -> depot). Depot is node 0."""
    if route.shape[0] == 0:
        return 0.0
    nodes = np.concatenate(([0], route, [0]))
    return dist_matrix[nodes[:-1], nodes[1:]].sum()

def solution_cost(routes: Solution, dist_matrix: DistanceMatrix) -> float:
    """Total distance across all routes."""
    return sum(route_cost(r, dist_matrix) for r in routes)

def is_feasible_cvrp(routes: Solution, instance: CvrpInstance) -> bool:
    """Check capacity constraint for each route."""
    for route in routes:
        if instance.demands[route].sum() > instance.capacity:
            return False
    return True

def is_feasible_vrptw(routes: Solution, instance: VrptwInstance) -> bool:
    """Check capacity + time window constraints for each route."""
    for route in routes:
        if instance.demands[route].sum() > instance.capacity:
            return False
        route_time = 0.0
        prev = 0
        for node in route:
            route_time += instance.dist_matrix[prev, node]
            if route_time > instance.tw_close[node]:
                return False
            route_time = max(route_time, instance.tw_open[node]) + instance.service_time[node]
            prev = node
    return True

### Local search operators

In [31]:
def two_opt(route: Route, dist_matrix: DistanceMatrix):
    """Intra-route 2-opt improvement."""
    pass

def or_opt(route: Route, dist_matrix: DistanceMatrix):
    """Intra-route Or-opt: relocate subsequences of 1, 2, or 3 customers."""
    pass

def relocate(routes: Solution, dist_matrix: DistanceMatrix):
    """Inter-route: move a customer from one route to another."""
    pass

def local_search(routes: Solution, instance: CvrpInstance | VrptwInstance):
    """Apply local search operators to improve a VRP solution."""
    pass

### Greedy randomized adaptive search procedure (GRASP)

In [32]:
def grasp(instance, max_iterations=100, alpha=0.3):
    pass

### Ant Colony Optimization (ACO)

In [33]:
def initialize_pheromones(n, tau_0=1.0):
    """Initialize pheromone matrix."""
    pass

def compute_heuristic_info(dist_matrix):
    """Heuristic information eta_ij = 1 / d_ij (visibility)."""
    pass

def ant_construct_solution(instance, pheromones, heuristic, alpha=1.0, beta=2.0):
    """
    Single ant constructs a full VRP solution.
    Builds routes one by one: picks next customer probabilistically
    based on pheromone * heuristic, starts new route when capacity
    (or time window for VRPTW) would be violated.
    """
    pass

def update_pheromones(pheromones, solutions, costs, rho=0.1):
    """
    Pheromone update: evaporation + deposit.
    rho: evaporation rate
    """
    pass

def aco(instance, n_ants=20, n_iterations=100, alpha=1.0, beta=2.0, rho=0.1):
    """
    ACO framework for VRP (works for both CVRP and VRPTW).
    
    Parameters:
        instance:     cvrp_instance or vrptw_instance
        n_ants:       number of ants per iteration
        n_iterations: number of iterations
        alpha:        pheromone importance
        beta:         heuristic importance
        rho:          evaporation rate
    
    Returns:
        best_routes, best_cost, convergence_history
    """
    pass

# Visualizations

# Results